# DiCE - Diverse Counterfactual Explanations

Counterfactual explanations answer "what would need to change for a different outcome?" DiCE generates diverse counterfactual examples — minimal changes to the input that would flip the model's prediction. This is particularly intuitive for end users.

In [ ]:
import dice_ml
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

In [ ]:
# Load the Adult Income dataset
dataset = dice_ml.datasets.DiCEDataset()
df = dataset.get_data()

# Display basic info about the dataset
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nOutcome distribution:")
print(df['income'].value_counts())

In [ ]:
# Prepare the data and train a model
# Separate features and target
X = df.drop('income', axis=1)
y = df['income']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a RandomForestClassifier
model = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Evaluate model
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"Training accuracy: {train_score:.4f}")
print(f"Testing accuracy: {test_score:.4f}")

# Create DiCE data object
d = dice_ml.Data(dataframe=df, continuous_features=['age', 'hours_per_week'], outcome_name='income')

# Create DiCE model wrapper
m = dice_ml.Model(model=model, backend='sklearn')

In [ ]:
# Create a DiCE explainer
exp = dice_ml.Dice(d, m)

# Select a query instance: a person predicted as <=50K
# Let's pick the first test instance
query_instance = X_test.iloc[0:1]
prediction = model.predict(query_instance)[0]
print(f"Query instance prediction: {prediction}")
print(f"\nQuery instance features:")
print(query_instance)

# Generate counterfactuals
counterfactuals = exp.generate_counterfactuals(query_instance, total_CFs=4, desired_class="opposite")
print(f"\nCounterfactuals generated successfully!")

In [ ]:
# Display the counterfactuals with only the changes shown
counterfactuals.visualize_as_dataframe(show_only_changes=True)

## Feature Constraints

We can constrain which features are allowed to change and specify their permitted ranges to generate more realistic counterfactuals.

In [ ]:
# Generate counterfactuals with feature constraints
# Example: age can only increase (people don't get younger), hours_per_week in realistic range
permitted_range = {'age': [30, 70], 'hours_per_week': [0, 60]}

counterfactuals_constrained = exp.generate_counterfactuals(
    query_instance, 
    total_CFs=4, 
    desired_class="opposite",
    permitted_range=permitted_range
)

print("Counterfactuals with feature constraints:")
counterfactuals_constrained.visualize_as_dataframe(show_only_changes=True)

## Interpretation

Each row shows minimal changes needed for the opposite prediction. Diversity ensures different actionable paths are shown (e.g., one counterfactual may increase education while another increases hours worked), giving users multiple strategies to achieve the desired outcome.